# find_clades_jaccard
For inferred clades which lack corresponding simulated clades, use the simulated clade with the highest jaccard index. 


In [176]:
import os

import itertools

#from Bio import SeqIO, AlignIO

#import gzip
import csv

import subprocess, gzip, tempfile
import shutil

from concurrent.futures import ProcessPoolExecutor, as_completed
import warnings

from pathlib import Path
from collections import defaultdict

import traceback

from tqdm.notebook import tqdm

In [177]:
#import tsinfer
import tskit
#import msprime
#import tsdate

import numpy as np
import pandas as pd

#import datetime as dt
#import time

#import matplotlib.pyplot as plt
#%matplotlib inline 

# from sklearn.linear_model import LinearRegression
from itertools import combinations
#import random 

In [178]:
os.getcwd()

'/home/nahmed/git/bacterial-args/migration/sweep_height_long/clades'

## define clades

In [179]:
def get_clades_sim(ts): 
    type = "sim"

    rows = []
    start = 0
    end   = ts.num_trees
    
    for i in range(start, end):
        t = ts.at_index(i)
        L, R = t.interval
        for u in t.nodes():
            if t.is_internal(u):
                height = t.time(u)
                pop = ts.population(ts.node(u).population).metadata["name"]
                leaves = tuple(sorted(t.get_leaves(u)))
                rows.append({
                #"index": i, 
                "left": L, 
                "right": R,
                "mrca_population": pop, 
                "type": type,
                #"kind": kind,
                "node": u,
                "node_height": height,
                "leaves": leaves
            })
        
    return pd.DataFrame(rows)

In [180]:
def get_clades_inf(ts): 
    type = 'inf' 
    
    rows = []
    start = 1
    end   = ts.num_trees - 1

    for i in range(start, end):
        t = ts.at_index(i)
        L, R = t.interval
        for u in t.nodes():
            if t.is_internal(u):
                height = t.time(u)
                leaves = tuple(sorted(t.get_leaves(u)))
                rows.append({
                #"index": i, 
                "left": L, 
                "right": R, 
                "type": type,
                #"kind": kind,
                "node": u,
                "node_height": height,
                "leaves": leaves
            })
        
    return pd.DataFrame(rows)

## binning 

In [181]:
def make_edges(L, bin_size):
    return np.arange(bin_size/2, int(L) +(bin_size/2), bin_size, dtype=np.int64)
    #return np.arange(0, int(L) + bin_size, bin_size, dtype=np.int64)

def add_bins(df, positions):
    out = []
    
    for i, pos in enumerate(positions):
        mask = (df["left"] <= pos) & (pos < df["right"])   # half-open [left, right) like tskit intervals
        if mask.any():
            tmp = df.loc[mask].copy()
            tmp["bin"] = i
            tmp["position"] = int(pos)
            out.append(tmp)
            
    return pd.concat(out, ignore_index=True)

## find matches between simulated and inferred clades

In [182]:
def match(row):
    if row['_merge'] == "both":
        return 1
    if row['_merge'] == "left_only":
        return 0
    if row['_merge'] == "right_only": 
        return -1

In [183]:
def match_clades(trees, outfile):

    sim = tskit.load("../trees/"+trees[1])
    inf = tskit.load("../trees/"+trees[0])

    positions = make_edges(L = 3e6, bin_size = 1e5)
    
    clades_sim = get_clades_sim(sim)
    clades_inf = get_clades_inf(inf)
    
    clades_sim_binned = add_bins(clades_sim, positions)
    clades_inf_binned = add_bins(clades_inf, positions)

    clades_sim_binned = clades_sim_binned.drop(columns=['left', 'right'])
    clades_inf_binned = clades_inf_binned.drop(columns=['left', 'right'])

    # check if matching sim node exists at same inf position, encode as [1]
    clades = pd.concat([clades_sim_binned,clades_inf_binned], axis=0, join='outer', ignore_index=False, keys=None)
    clades['match'] = clades.duplicated(subset=['leaves', 'bin', 'position'], keep=False).astype(int)

    res_sim = clades[clades['type'] == 'sim'] 
    res_inf = get_jaccard_matches(clades) 

    res_sim.to_csv('jaccard/sim'+outfile, index = False)
    res_inf.to_csv('jaccard/inf'+outfile, index = False)

    return clades 

    #clades.to_csv(outfile, index=False)

In [191]:
clades

,mrca_population,type,node,node_height,leaves,bin,position,match
0,pop_0,sim,553,8747.914309,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",0,50000,1
1,pop_0,sim,243,1124.519129,"(6, 14, 33, 35, 36, 38)",0,50000,1
2,pop_2,sim,56,35.621751,"(33, 35, 36, 38)",0,50000,1
3,pop_2,sim,44,5.166702,"(35, 38)",0,50000,1
4,pop_2,sim,45,6.580362,"(33, 36)",0,50000,0
...,...,...,...,...,...,...,...,...
1021,NaN,inf,255,131.880234,"(17, 18, 19, 21, 27, 28, 29, 30)",29,2950000,1
1022,NaN,inf,140,17.030842,"(17, 18, 19, 21)",29,2950000,1
1023,NaN,inf,118,3.361342,"(17, 18, 21)",29,2950000,1
1024,NaN,inf,155,27.378888,"(27, 28, 29, 30)",29,2950000,1


### jaccard index
measures the similarity between two sets by dividing the number of unique shared items (intersection) by the total number of unique items across both sets (union)

In [184]:
# find jaccard distance for (inferred clade, simulated clade) 
def jdist(row):
    s1 = set(row['leaves_inf'])
    s2 = set(row['leaves_sim'])
    
    intersection = len(s1 & s2) 
    union = len(s1 | s2) 
    #print(intersection)

    if union == 0:
        return 0 
    
    return intersection/union 

In [185]:
# simulated matches conditional on containing every individual in the inferred clade 
def check_leaves(row):
    return len(list(set(row['leaves_inf']))) == len(list(set(row['leaves_inf']) & set(row['leaves_sim'])))

In [186]:
def get_jaccard_matches(clades): 
    bci = clades[(clades['type'] == 'inf') & (clades['match'] == 0)]
    bcs = clades[clades['type'] == 'sim'] 
    cross = pd.merge(bci, bcs, on=['bin', 'position'], how='inner', suffixes=['_inf', '_sim'])
    cross['jaccard'] = cross.apply(jdist, axis=1)
    mask = cross.apply(check_leaves, axis=1)
    cross_masked = cross[mask]
    top = cross_masked.groupby(['bin', 'position', 'node_inf'])['jaccard'].transform(max) == cross_masked['jaccard']
    cross_top = cross_masked[top]
    cross_rename = cross_top.rename(columns={'mrca_population_inf': 'mrca_population', 
                                             'type_inf':'type', 
                                             'node_inf': 'node',
                                             'node_height_inf': 'node_height', 
                                             'leaves_inf':'leaves', 
                                             'match_inf':'match'})
    cross_final = cross_rename[['mrca_population', 'type', 'node', 'node_height', 'leaves', 'bin', 'position', 'match', 'node_sim', 'leaves_sim', 'jaccard']]
    res = pd.concat([clades[(clades['type'] == 'inf') & (clades['match'] == 1)], cross_final], ignore_index=True)
    
    return res 
    

# apply all (in parallel) 

In [187]:
def find_clades_parallel(matches, max_workers):

    desc = f"Finding clades"

    # build jobs
    jobs = []
    job_idx = 0

    for trees, outfile in zip(matches.values(), matches.keys()):
        jobs.append((trees, outfile))
        job_idx += 1

    results = []
    failures = []

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                match_clades,
                trees = trees,
                outfile=outfile
            ): (trees, outfile)
            for (trees, outfile) in jobs
        }

        for fut in tqdm(as_completed(futures), total=len(futures), desc="find_clades"):
            job = futures[fut]
            try:
                fut.result()
                results.append(job)
            except Exception:
                failures.append({
                    "job": job,
                    "traceback": traceback.format_exc()
                })

    return results, failures

In [188]:
folder = Path("../trees")
groups = defaultdict(list)

for file_path in folder.glob('*'):
    if file_path.is_file():
        # characters after sim_ or inf_ prefix 
        suffix = '_clades_' + file_path.name[4:-6] + '.csv' 
        groups[suffix].append(file_path.name)
        groups[suffix] = sorted(groups[suffix])

# pairs only  
matches = {k: v for k, v in groups.items() if len(v) > 1}

In [189]:
results, failures = find_clades_parallel(
    matches,
    max_workers= max(1, (os.cpu_count() or 2) - 1)
)

find_clades:   0%|          | 0/609 [00:00<?, ?it/s]

In [190]:
#test files 
#clades = match_clades(matches['_clades_ss_mu2.508e-8_pm3.000e-1_mr5.000e-3_seed511.csv'], 'clades_ss_mu2.508e-8_pm3.000e-1_mr5.000e-3_seed511.csv') 